# 🔍 GitHub Account Audit — Court Overview & LLM Pipeline

**Purpose:** Build a complete, timestamped record of all GitHub activity across all repositories.  
**Structure:** Each section is self-contained and checkpoints to disk — safe to re-run after interruptions (GitHub API rate limits are 5 000 req/h).  
**Output:** `github_audit/` directory with CSVs, JSONs, and an LLM-ready commit bundle.

| Section | What it collects |
|---------|------------------|
| 0 | Setup, auth, helpers |
| 1 | Account snapshot (profile, key dates) |
| 2 | Repository inventory (all repos incl. forks, archived) |
| 3 | Commit extraction (all commits, all repos) |
| 4 | File-level diffs per commit |
| 5 | Issues, PRs, labels, comments |
| 6 | Account events stream (all event types) |
| 7 | Gists |
| 8 | Organization memberships |
| 9 | Spam investigation — pattern analysis |
| 10 | LLM-ready export (one JSON per commit) |
| 11 | Summary report (court-ready Markdown) |

## Section 0 — Setup, Auth & Helpers

In [ ]:
# Install dependencies
!pip install PyGithub pandas tqdm matplotlib seaborn python-dotenv requests ipywidgets --quiet

In [ ]:
import os
import json
import time
import hashlib
import warnings
from datetime import datetime, timezone, timedelta
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
from tqdm.notebook import tqdm
from github import Github, GithubException, RateLimitExceededException
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# ─── CONFIGURE HERE ──────────────────────────────────────────────────────────
# Create a Personal Access Token at https://github.com/settings/tokens
# Scopes needed: repo (full), read:user, read:org, gist
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN") or "ghp_YOUR_TOKEN_HERE"

# Leave empty to audit your own account, or set to another username
TARGET_USER = ""  # e.g. "octocat" or leave blank

# Set to True to include private repos (requires appropriate token scopes)
INCLUDE_PRIVATE = True

# Max diffs to store per commit (bytes). Large diffs are truncated for LLM.
MAX_PATCH_BYTES = 8000
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("github_audit")
OUTPUT_DIR.mkdir(exist_ok=True)
for sub in ["commits", "diffs", "events", "issues", "llm_bundles", "gists"]:
    (OUTPUT_DIR / sub).mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# ─── GitHub client & helpers ──────────────────────────────────────────────────

g = Github(GITHUB_TOKEN, per_page=100, retry=3)
me = g.get_user()
TARGET = g.get_user(TARGET_USER) if TARGET_USER else me

print(f"Authenticated as : {me.login}")
print(f"Auditing account : {TARGET.login}")
print(f"Account created  : {TARGET.created_at}")
rl = g.get_rate_limit()
print(f"Rate limit       : {rl.core.remaining}/{rl.core.limit} requests remaining")
print(f"Resets at        : {rl.core.reset}")


def rate_limit_wait(buffer: int = 100):
    """Sleep until rate limit resets if fewer than `buffer` requests remain."""
    rl = g.get_rate_limit()
    if rl.core.remaining < buffer:
        reset_ts = rl.core.reset.replace(tzinfo=timezone.utc)
        wait_secs = (reset_ts - datetime.now(timezone.utc)).total_seconds() + 5
        print(f"  ⏳ Rate limit low ({rl.core.remaining} left). Sleeping {wait_secs:.0f}s …")
        time.sleep(max(wait_secs, 0))


def safe_paginate(paginated_list, desc="", limit=None):
    """Iterate a PyGitHub PaginatedList with rate-limit awareness."""
    items = []
    for i, item in enumerate(tqdm(paginated_list, desc=desc)):
        rate_limit_wait()
        items.append(item)
        if limit and i + 1 >= limit:
            break
    return items


def ts(dt) -> str:
    """Normalise a datetime to ISO-8601 UTC string, or empty string if None."""
    if dt is None:
        return ""
    if hasattr(dt, "isoformat"):
        return dt.isoformat()
    return str(dt)


def save_json(data, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, default=str)


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


print("\n✅ Client initialised.")

## Section 1 — Account Snapshot

Everything that describes the account itself — the "cover page" of your court exhibit.

In [ ]:
account_snapshot = {
    "login":              TARGET.login,
    "id":                 TARGET.id,
    "name":               TARGET.name,
    "email":              TARGET.email,
    "bio":                TARGET.bio,
    "company":            TARGET.company,
    "location":           TARGET.location,
    "blog":               TARGET.blog,
    "twitter_username":   TARGET.twitter_username if hasattr(TARGET, 'twitter_username') else None,
    "account_type":       TARGET.type,             # User vs Organization
    "created_at":         ts(TARGET.created_at),
    "updated_at":         ts(TARGET.updated_at),
    "public_repos":       TARGET.public_repos,
    "public_gists":       TARGET.public_gists,
    "followers":          TARGET.followers,
    "following":          TARGET.following,
    "site_admin":         TARGET.site_admin,
    # Private fields only visible on own account with proper token:
    "total_private_repos": getattr(TARGET, 'total_private_repos', None),
    "owned_private_repos": getattr(TARGET, 'owned_private_repos', None),
    "disk_usage_kb":       getattr(TARGET, 'disk_usage', None),
    "two_factor_enabled":  getattr(TARGET, 'two_factor_authentication', None),
    "plan":                getattr(getattr(TARGET, 'plan', None), 'name', None),
    "audit_generated_at": ts(datetime.now(timezone.utc)),
}

save_json(account_snapshot, OUTPUT_DIR / "account_snapshot.json")

print("Account Snapshot")
print("=" * 60)
for k, v in account_snapshot.items():
    if v is not None:
        print(f"  {k:30s}: {v}")

## Section 2 — Repository Inventory

All repos the account owns or has forked — including archived and (if token allows) private ones.

In [ ]:
CHECKPOINT_REPOS = OUTPUT_DIR / "repos.csv"

if CHECKPOINT_REPOS.exists():
    print(f"📂 Loading checkpoint: {CHECKPOINT_REPOS}")
    repos_df = pd.read_csv(CHECKPOINT_REPOS)
    print(f"   {len(repos_df)} repos loaded.")
else:
    repo_type = "all" if INCLUDE_PRIVATE else "public"
    raw_repos = safe_paginate(
        TARGET.get_repos(type=repo_type, sort="created", direction="asc"),
        desc="📦 Fetching repos"
    )

    repo_records = []
    for r in tqdm(raw_repos, desc="🔍 Extracting repo metadata"):
        rate_limit_wait()
        try:
            languages = dict(r.get_languages())  # {lang: bytes}
            primary_language = max(languages, key=languages.get) if languages else None
        except GithubException:
            primary_language = r.language
            languages = {}

        try:
            topics = r.get_topics()
        except GithubException:
            topics = []

        repo_records.append({
            "repo_full_name":   r.full_name,
            "repo_name":        r.name,
            "description":      r.description,
            "private":          r.private,
            "fork":             r.fork,
            "archived":         r.archived,
            "disabled":         r.disabled if hasattr(r, 'disabled') else None,
            "created_at":       ts(r.created_at),
            "updated_at":       ts(r.updated_at),
            "pushed_at":        ts(r.pushed_at),
            "primary_language": primary_language,
            "languages_json":   json.dumps(languages),
            "topics":           ",".join(topics),
            "default_branch":   r.default_branch,
            "size_kb":          r.size,
            "stars":            r.stargazers_count,
            "forks":            r.forks_count,
            "watchers":         r.watchers_count,
            "open_issues":      r.open_issues_count,
            "has_issues":       r.has_issues,
            "has_wiki":         r.has_wiki,
            "has_projects":     r.has_projects if hasattr(r, 'has_projects') else None,
            "license":          r.license.name if r.license else None,
            "visibility":       r.visibility if hasattr(r, 'visibility') else ("private" if r.private else "public"),
            "clone_url":        r.clone_url,
            "parent_repo":      r.parent.full_name if r.fork and r.parent else None,
            "source_repo":      r.source.full_name if r.fork and r.source else None,
        })

    repos_df = pd.DataFrame(repo_records)
    repos_df.to_csv(CHECKPOINT_REPOS, index=False)
    print(f"\n✅ {len(repos_df)} repos saved to {CHECKPOINT_REPOS}")

# Quick summary
print("\nRepository breakdown:")
print(repos_df[["private", "fork", "archived"]].value_counts().to_string())
print(f"\nLanguages: {repos_df['primary_language'].value_counts().to_dict()}")

## Section 3 — Commit Extraction

Every commit across every repository: author, date, message, and stats.  
**This is the most API-intensive step** — checkpoints after each repo.

In [ ]:
CHECKPOINT_COMMITS = OUTPUT_DIR / "commits" / "all_commits.csv"
CHECKPOINT_DONE_REPOS = OUTPUT_DIR / "commits" / "done_repos.json"

# Load existing checkpoint
if CHECKPOINT_COMMITS.exists():
    commits_df = pd.read_csv(CHECKPOINT_COMMITS)
    done_repos = load_json(CHECKPOINT_DONE_REPOS) if CHECKPOINT_DONE_REPOS.exists() else []
    print(f"📂 Checkpoint: {len(commits_df)} commits already collected from {len(done_repos)} repos.")
else:
    commits_df = pd.DataFrame()
    done_repos = []

all_commit_records = commits_df.to_dict('records') if not commits_df.empty else []
pending_repos = [r for r in repos_df['repo_full_name'].tolist() if r not in done_repos]
print(f"Repos remaining : {len(pending_repos)}")


for repo_name in tqdm(pending_repos, desc="📦 Repos"):
    rate_limit_wait()
    try:
        repo = g.get_repo(repo_name)
    except GithubException as e:
        print(f"  ⚠️  Cannot access {repo_name}: {e.status}")
        done_repos.append(repo_name)
        continue

    try:
        commits_iter = repo.get_commits(author=TARGET.login)
        for commit in tqdm(commits_iter, desc=f"  commits [{repo_name}]", leave=False):
            rate_limit_wait()
            try:
                c = commit.commit
                author_obj = c.author
                stats = commit.stats  # triggers extra API call — contains additions/deletions

                all_commit_records.append({
                    # Identity
                    "repo":             repo_name,
                    "sha":              commit.sha,
                    "short_sha":        commit.sha[:7],
                    # Author info (from git object — may differ from GitHub account)
                    "git_author_name":  author_obj.name if author_obj else None,
                    "git_author_email": author_obj.email if author_obj else None,
                    "git_author_date":  ts(author_obj.date) if author_obj else None,
                    # Committer (may differ if commit was amended or squashed)
                    "git_committer_name":  c.committer.name if c.committer else None,
                    "git_committer_email": c.committer.email if c.committer else None,
                    "git_committer_date":  ts(c.committer.date) if c.committer else None,
                    # GitHub account linked to the commit
                    "gh_author_login":  commit.author.login if commit.author else None,
                    # Message
                    "message":          c.message,
                    "message_first_line": c.message.split("\n")[0].strip(),
                    # Stats
                    "additions":        stats.additions,
                    "deletions":        stats.deletions,
                    "total_changes":    stats.total,
                    "files_changed":    len(commit.files),
                    # Verification (important for court — proves commits were signed)
                    "verified":         c.raw_data.get('verification', {}).get('verified', False),
                    "verification_reason": c.raw_data.get('verification', {}).get('reason', ''),
                    # Tree & parent info
                    "tree_sha":         c.tree.sha if c.tree else None,
                    "parent_count":     len(commit.parents),
                    "parent_shas":      ",".join([p.sha for p in commit.parents]),
                    "is_merge_commit":  len(commit.parents) > 1,
                    # Links
                    "html_url":         commit.html_url,
                    "comments_count":   c.comment_count,
                })
            except GithubException as e:
                print(f"    ⚠️  Skip commit {commit.sha[:7]}: {e.status}")

    except GithubException as e:
        print(f"  ⚠️  Error iterating commits for {repo_name}: {e.status}")

    # Checkpoint after each repo
    done_repos.append(repo_name)
    pd.DataFrame(all_commit_records).to_csv(CHECKPOINT_COMMITS, index=False)
    save_json(done_repos, CHECKPOINT_DONE_REPOS)

commits_df = pd.DataFrame(all_commit_records)
commits_df['git_author_date'] = pd.to_datetime(commits_df['git_author_date'], utc=True, errors='coerce')
commits_df = commits_df.sort_values('git_author_date')

print(f"\n✅ Total commits collected: {len(commits_df)}")
print(f"   Date range: {commits_df['git_author_date'].min()} → {commits_df['git_author_date'].max()}")

## Section 4 — File-Level Diffs per Commit

For each commit: which files changed, how, and the actual patch.  
Stored individually per commit SHA (avoids re-fetching on restart).

In [ ]:
DIFF_DIR = OUTPUT_DIR / "diffs"

already_done_shas = {p.stem for p in DIFF_DIR.glob("*.json")}
pending_commits = commits_df[~commits_df['sha'].isin(already_done_shas)]

print(f"Diffs already fetched : {len(already_done_shas)}")
print(f"Diffs remaining       : {len(pending_commits)}")


def extract_file_diff(commit_file) -> dict:
    patch = getattr(commit_file, 'patch', None) or ""
    # Truncate very large patches to stay LLM-friendly
    truncated = False
    if len(patch.encode()) > MAX_PATCH_BYTES:
        patch = patch.encode()[:MAX_PATCH_BYTES].decode('utf-8', errors='ignore') + "\n… [TRUNCATED]"
        truncated = True
    return {
        "filename":          commit_file.filename,
        "status":            commit_file.status,        # added / modified / deleted / renamed
        "additions":         commit_file.additions,
        "deletions":         commit_file.deletions,
        "changes":           commit_file.changes,
        "previous_filename": getattr(commit_file, 'previous_filename', None),
        "blob_url":          commit_file.blob_url,
        "raw_url":           commit_file.raw_url,
        "patch":             patch,
        "patch_truncated":   truncated,
    }


for _, row in tqdm(pending_commits.iterrows(), total=len(pending_commits), desc="🗂️  Fetching diffs"):
    rate_limit_wait()
    sha = row['sha']
    try:
        repo = g.get_repo(row['repo'])
        commit = repo.get_commit(sha)
        files = [extract_file_diff(f) for f in commit.files]
        diff_record = {
            "sha":   sha,
            "repo":  row['repo'],
            "date":  row['git_author_date'].isoformat() if pd.notna(row['git_author_date']) else "",
            "files": files,
        }
        save_json(diff_record, DIFF_DIR / f"{sha}.json")
    except GithubException as e:
        print(f"  ⚠️  {sha[:7]}: {e.status}")

print(f"\n✅ Diffs stored in {DIFF_DIR}")

## Section 5 — Issues, PRs, Labels & Comments

All issues and pull requests, their labels, and every associated comment.

In [ ]:
CHECKPOINT_ISSUES = OUTPUT_DIR / "issues" / "all_issues.csv"
CHECKPOINT_COMMENTS = OUTPUT_DIR / "issues" / "all_comments.csv"

issue_records = []
comment_records = []

for repo_name in tqdm(repos_df['repo_full_name'].tolist(), desc="📋 Issues & PRs"):
    rate_limit_wait()
    try:
        repo = g.get_repo(repo_name)
        issues = repo.get_issues(state='all', creator=TARGET.login, sort='created', direction='asc')

        for issue in tqdm(issues, desc=f"  [{repo_name}]", leave=False):
            rate_limit_wait()
            is_pr = issue.pull_request is not None

            issue_records.append({
                "repo":         repo_name,
                "number":       issue.number,
                "type":         "pull_request" if is_pr else "issue",
                "title":        issue.title,
                "body":         (issue.body or "")[:2000],  # truncate for CSV
                "state":        issue.state,
                "labels":       ",".join([l.name for l in issue.labels]),
                "assignees":    ",".join([a.login for a in issue.assignees]),
                "created_at":   ts(issue.created_at),
                "updated_at":   ts(issue.updated_at),
                "closed_at":    ts(issue.closed_at),
                "comments":     issue.comments,
                "html_url":     issue.html_url,
            })

            # Fetch all comments on this issue/PR
            try:
                for comment in issue.get_comments():
                    comment_records.append({
                        "repo":        repo_name,
                        "issue_number": issue.number,
                        "comment_id":  comment.id,
                        "author":      comment.user.login if comment.user else None,
                        "body":        (comment.body or "")[:2000],
                        "created_at":  ts(comment.created_at),
                        "updated_at":  ts(comment.updated_at),
                        "html_url":    comment.html_url,
                    })
            except GithubException:
                pass

    except GithubException as e:
        print(f"  ⚠️  {repo_name}: {e.status}")

# Also collect commit-level comments (separate from issue comments)
print("\nFetching commit comments …")
commit_comment_records = []
for repo_name in tqdm(repos_df['repo_full_name'].tolist(), desc="💬 Commit comments"):
    rate_limit_wait()
    try:
        repo = g.get_repo(repo_name)
        for cc in repo.get_comments():
            if cc.user and cc.user.login == TARGET.login:
                commit_comment_records.append({
                    "repo":       repo_name,
                    "comment_id": cc.id,
                    "commit_sha": cc.commit_id,
                    "author":     cc.user.login,
                    "path":       cc.path,
                    "position":   cc.position,
                    "line":       cc.line,
                    "body":       (cc.body or "")[:2000],
                    "created_at": ts(cc.created_at),
                    "html_url":   cc.html_url,
                })
    except GithubException:
        pass

issues_df = pd.DataFrame(issue_records)
comments_df = pd.DataFrame(comment_records)
commit_comments_df = pd.DataFrame(commit_comment_records)

issues_df.to_csv(CHECKPOINT_ISSUES, index=False)
comments_df.to_csv(OUTPUT_DIR / "issues" / "all_issue_comments.csv", index=False)
commit_comments_df.to_csv(OUTPUT_DIR / "issues" / "all_commit_comments.csv", index=False)

print(f"\n✅ Issues/PRs: {len(issues_df)}")
print(f"   Issue comments: {len(comments_df)}")
print(f"   Commit comments: {len(commit_comments_df)}")

## Section 6 — Account Events Stream

GitHub's Events API captures all public activity (limited to ~90 days of history, ~300 events).  
This covers: pushes, issue actions, PR actions, stars, forks, repo creation/deletion, gist creation, etc.

In [ ]:
EVENTS_FILE = OUTPUT_DIR / "events" / "account_events.json"

# PyGitHub wraps the events API
events_raw = []
for event in tqdm(TARGET.get_events(), desc="⚡ Account events"):
    rate_limit_wait()
    events_raw.append({
        "id":           event.id,
        "type":         event.type,   # PushEvent, CreateEvent, IssuesEvent …
        "actor":        event.actor.login if event.actor else None,
        "repo":         event.repo.name if event.repo else None,
        "public":       event.public,
        "created_at":   ts(event.created_at),
        "payload":      event.raw_data.get('payload', {}),
    })

save_json(events_raw, EVENTS_FILE)

events_df = pd.DataFrame(events_raw)
events_df.to_csv(OUTPUT_DIR / "events" / "account_events.csv", index=False)

print(f"✅ {len(events_raw)} events captured.")
if len(events_raw):
    print("\nEvent type breakdown:")
    print(events_df['type'].value_counts().to_string())

## Section 7 — Gists

Often overlooked — gists are a separate content layer on GitHub and part of your activity record.

In [ ]:
GIST_FILE = OUTPUT_DIR / "gists" / "all_gists.json"

gist_records = []
for gist in tqdm(TARGET.get_gists(), desc="📝 Gists"):
    rate_limit_wait()
    gist_records.append({
        "id":           gist.id,
        "description":  gist.description,
        "public":       gist.public,
        "created_at":   ts(gist.created_at),
        "updated_at":   ts(gist.updated_at),
        "files":        list(gist.files.keys()),
        "comments":     gist.comments,
        "forks":        gist.forks,
        "html_url":     gist.html_url,
        "git_pull_url": gist.git_pull_url,
    })

save_json(gist_records, GIST_FILE)
pd.DataFrame(gist_records).to_csv(OUTPUT_DIR / "gists" / "all_gists.csv", index=False)
print(f"✅ {len(gist_records)} gists captured.")

## Section 8 — Organization Memberships

In [ ]:
org_records = []
try:
    for org in tqdm(TARGET.get_orgs(), desc="🏢 Orgs"):
        rate_limit_wait()
        org_records.append({
            "login":       org.login,
            "name":        org.name,
            "description": org.description,
            "public_repos": org.public_repos,
            "created_at":  ts(org.created_at),
            "html_url":    org.html_url,
        })
except GithubException as e:
    print(f"  ⚠️  Cannot list orgs: {e.status}")

orgs_df = pd.DataFrame(org_records)
orgs_df.to_csv(OUTPUT_DIR / "organizations.csv", index=False)
print(f"✅ {len(org_records)} organization memberships.")
if org_records:
    print(orgs_df[['login', 'name', 'created_at']].to_string(index=False))

## Section 9 — Spam Investigation: Pattern Analysis

Think of spam detectors like airport metal detectors — they flag anything that matches a *pattern*, not just actual weapons. This section looks for the patterns that could have triggered a spam flag:

- **Commit bursts** (many commits in a short window)
- **Regularity** (machine-like fixed intervals between commits — bots are too regular)
- **Email diversity** (commits from multiple email addresses)
- **Repo creation spikes** (many repos created in a short period)
- **Message repetition** (same or near-identical commit messages)
- **Push event anomalies** (unusual push sizes or frequencies)

In [ ]:
# Reload commits if needed
if 'commits_df' not in dir() or commits_df.empty:
    commits_df = pd.read_csv(CHECKPOINT_COMMITS)
    commits_df['git_author_date'] = pd.to_datetime(commits_df['git_author_date'], utc=True, errors='coerce')

df = commits_df.copy()
df = df.dropna(subset=['git_author_date'])
df = df.sort_values('git_author_date')

# ── 9.1 Timeline overview ──────────────────────────────────────────────────
df['date'] = df['git_author_date'].dt.date
df['hour'] = df['git_author_date'].dt.hour
df['weekday'] = df['git_author_date'].dt.day_name()
df['week'] = df['git_author_date'].dt.to_period('W')

commits_per_day = df.groupby('date').size()
peak_day = commits_per_day.idxmax()
peak_count = commits_per_day.max()

print("=" * 60)
print("COMMIT PATTERN ANALYSIS")
print("=" * 60)
print(f"Total commits analysed : {len(df)}")
print(f"Date range             : {df['git_author_date'].min().date()} → {df['git_author_date'].max().date()}")
print(f"Active days            : {df['date'].nunique()}")
print(f"Peak single day        : {peak_day} ({peak_count} commits)")
print(f"Average commits/day    : {commits_per_day.mean():.1f}")

# Burst windows: days with unusually high commit count (>2σ above mean)
mean_c = commits_per_day.mean()
std_c = commits_per_day.std()
burst_days = commits_per_day[commits_per_day > mean_c + 2 * std_c]
print(f"\nBurst days (>mean+2σ, potential spam triggers):")
for day, count in burst_days.sort_values(ascending=False).items():
    print(f"  {day}: {count} commits  ← z-score {(count-mean_c)/std_c:.1f}")

In [ ]:
# ── 9.2 Email diversity ────────────────────────────────────────────────────
print("\nEmail addresses used in commits:")
email_counts = df['git_author_email'].value_counts()
for email, count in email_counts.items():
    print(f"  {email:45s}  {count:4d} commits")

# ── 9.3 Commit message analysis ───────────────────────────────────────────
print("\nTop 20 most-repeated first-line commit messages:")
msg_counts = df['message_first_line'].value_counts().head(20)
for msg, count in msg_counts.items():
    flag = " ⚠️  REPEATED" if count > 5 else ""
    print(f"  [{count:3d}x]  {msg[:80]}{flag}")

# ── 9.4 Commit interval regularity (bot detection) ───────────────────────
print("\nCommit interval analysis (bots have suspiciously regular intervals):")
intervals_sec = df['git_author_date'].diff().dt.total_seconds().dropna()
intervals_min = intervals_sec / 60
print(f"  Median interval  : {intervals_min.median():.1f} min")
print(f"  Std of interval  : {intervals_min.std():.1f} min")
print(f"  Coefficient of variation (CV): {intervals_min.std()/intervals_min.mean():.2f}  "
      "(low CV = very regular = more bot-like)")

# ── 9.5 Repo creation timeline ────────────────────────────────────────────
print("\nRepository creation timeline:")
if 'repos_df' in dir():
    rdf = repos_df.copy()
    rdf['created_at'] = pd.to_datetime(rdf['created_at'], utc=True, errors='coerce')
    rdf['created_date'] = rdf['created_at'].dt.date
    repos_per_day = rdf.groupby('created_date').size()
    burst_repos = repos_per_day[repos_per_day >= 3]
    if not burst_repos.empty:
        print("  Days with 3+ repos created (potential spam trigger):")
        for day, cnt in burst_repos.items():
            print(f"    {day}: {cnt} repos created")
    else:
        print("  No repo creation bursts detected.")

In [ ]:
# ── 9.6 Visual: Activity heatmap ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"GitHub Activity Analysis: {TARGET.login}", fontsize=14, y=1.01)

# Commits per month
ax = axes[0, 0]
monthly = df.set_index('git_author_date').resample('ME').size()
ax.bar(monthly.index, monthly.values, width=20, color='steelblue', alpha=0.8)
ax.set_title('Commits per Month')
ax.set_xlabel('Date')
ax.set_ylabel('Commits')
ax.tick_params(axis='x', rotation=45)

# Commits by hour of day
ax = axes[0, 1]
hour_counts = df['hour'].value_counts().sort_index()
ax.bar(hour_counts.index, hour_counts.values, color='darkorange', alpha=0.8)
ax.set_title('Commits by Hour of Day (UTC)')
ax.set_xlabel('Hour (0–23)')
ax.set_ylabel('Commits')
ax.set_xticks(range(0, 24, 2))

# Commits by weekday
ax = axes[1, 0]
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df['weekday'].value_counts().reindex(day_order, fill_value=0)
colors = ['#2196F3' if d not in ['Saturday','Sunday'] else '#FF7043' for d in day_order]
ax.bar(day_counts.index, day_counts.values, color=colors, alpha=0.8)
ax.set_title('Commits by Weekday (blue=weekday, red=weekend)')
ax.tick_params(axis='x', rotation=30)

# Top 10 repos by commit count
ax = axes[1, 1]
top_repos = df['repo'].value_counts().head(10)
ax.barh(top_repos.index[::-1], top_repos.values[::-1], color='mediumseagreen', alpha=0.8)
ax.set_title('Top 10 Repos by Commit Count')
ax.set_xlabel('Commits')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "activity_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved to github_audit/activity_analysis.png")

In [ ]:
# ── 9.7 Spam flag investigation timeline ──────────────────────────────────
# GitHub does not expose the exact spam-flag timestamp via API.
# We triangulate it from surrounding signals.

print("=" * 60)
print("SPAM FLAG INVESTIGATION")
print("=" * 60)
print()
print("GitHub's spam detection is not exposed via API.")
print("The following signals help triangulate when it may have been triggered:\n")

# Look for sudden drops in activity (account restrictions reduce contribution graph)
weekly_commits = df.set_index('git_author_date').resample('W').size()
weekly_commits_pct_change = weekly_commits.pct_change()

drops = weekly_commits_pct_change[weekly_commits_pct_change < -0.6]
if not drops.empty:
    print("Weeks with >60% commit drop (account restriction candidate periods):")
    for period, pct in drops.items():
        before = weekly_commits.get(period - pd.Timedelta(weeks=1), 0)
        after = weekly_commits.get(period, 0)
        print(f"  Week ending {period.date()}: {before:.0f} → {after:.0f} commits ({pct*100:+.0f}%)")
else:
    print("  No sharp commit drops detected.")

print()
print("Check these manually on your GitHub account:")
print("  1. https://github.com/settings/profile  → look for 'flagged' banner")
print("  2. https://github.com/contact           → support ticket history")
print("  3. Check for emails from GitHub noreply around the suspected dates")
print("  4. Review contribution graph gap at https://github.com/{TARGET.login}")

# Document the context window around burst days for court
if not burst_days.empty:
    print(f"\nContext around peak burst day ({peak_day}):")
    window = df[
        (df['date'] >= (peak_day - pd.Timedelta(days=3))) &
        (df['date'] <= (peak_day + pd.Timedelta(days=3)))
    ][['repo', 'short_sha', 'git_author_date', 'message_first_line', 'additions', 'deletions']]
    print(window.to_string(index=False))

## Section 10 — LLM-Ready Export

Each commit is bundled into a self-contained JSON that an LLM can read to explain what changed and why.  
Think of it as writing a **"dossier"** for each commit that a future AI reviewer can pick up without any other context.

In [ ]:
LLM_DIR = OUTPUT_DIR / "llm_bundles"

already_bundled = {p.stem for p in LLM_DIR.glob("*.json")}
pending_for_bundle = commits_df[~commits_df['sha'].isin(already_bundled)]
print(f"Bundles already created : {len(already_bundled)}")
print(f"Bundles to create       : {len(pending_for_bundle)}")

# System prompt template for the LLM analysis step
LLM_SYSTEM_PROMPT = """\
You are a software archaeologist analysing a specific git commit.
You will receive a JSON object describing the commit and its file diffs.
Your task is to produce a structured explanation with these fields:

1. summary: One sentence describing the purpose of this commit.
2. intent: What the developer was trying to accomplish.
3. files_changed_summary: For each file, one sentence on what changed and why.
4. technical_notes: Any noteworthy technical patterns (refactors, bug fixes, new features, etc.).
5. red_flags: Anything unusual — force pushes, deleted files, large additions, non-standard patterns.
6. context_notes: What this commit suggests about the project's state at this point in time.

Respond ONLY in valid JSON. No markdown.
"""

for _, row in tqdm(pending_for_bundle.iterrows(), total=len(pending_for_bundle), desc="📦 Building LLM bundles"):
    sha = row['sha']
    diff_path = DIFF_DIR / f"{sha}.json"

    diff_data = load_json(diff_path) if diff_path.exists() else {"files": []}

    bundle = {
        # ── Metadata for court record ──────────────────────────────────
        "_meta": {
            "generated_at":      ts(datetime.now(timezone.utc)),
            "llm_system_prompt": LLM_SYSTEM_PROMPT,
        },
        # ── Commit identity ────────────────────────────────────────────
        "sha":              sha,
        "short_sha":        sha[:7],
        "html_url":         row.get('html_url', ''),
        "repo":             row['repo'],
        # ── Authorship (critical for court — who wrote this?) ──────────
        "author": {
            "git_name":     row.get('git_author_name'),
            "git_email":    row.get('git_author_email'),
            "git_date":     row.get('git_author_date') if pd.notna(row.get('git_author_date')) else None,
            "github_login": row.get('gh_author_login'),
        },
        "committer": {
            "git_name":  row.get('git_committer_name'),
            "git_email": row.get('git_committer_email'),
            "git_date":  row.get('git_committer_date'),
        },
        # ── Verification ───────────────────────────────────────────────
        "verified":            bool(row.get('verified', False)),
        "verification_reason": row.get('verification_reason', ''),
        # ── Commit message ─────────────────────────────────────────────
        "message":             row.get('message', ''),
        "message_first_line":  row.get('message_first_line', ''),
        # ── Stats summary ──────────────────────────────────────────────
        "stats": {
            "additions":     int(row.get('additions', 0)),
            "deletions":     int(row.get('deletions', 0)),
            "total_changes": int(row.get('total_changes', 0)),
            "files_changed": int(row.get('files_changed', 0)),
        },
        "is_merge_commit":  bool(row.get('is_merge_commit', False)),
        "parent_count":     int(row.get('parent_count', 1)),
        "parent_shas":      str(row.get('parent_shas', '')).split(','),
        # ── File diffs (the core content for LLM analysis) ────────────
        "files":            diff_data.get('files', []),
    }

    save_json(bundle, LLM_DIR / f"{sha}.json")

print(f"\n✅ LLM bundles saved to {LLM_DIR}")
print(f"   Total bundles: {len(list(LLM_DIR.glob('*.json')))}")

In [ ]:
# ── Optional: Run LLM analysis on a single commit (demo) ──────────────────
# Set a real SHA here to test the LLM pipeline on one commit before running bulk.

DEMO_SHA = None  # e.g. "abc1234def5678" — leave None to skip

if DEMO_SHA:
    import anthropic
    # pip install anthropic  (uses ANTHROPIC_API_KEY env var)
    client = anthropic.Anthropic()

    bundle_path = LLM_DIR / f"{DEMO_SHA}.json"
    if not bundle_path.exists():
        print(f"Bundle not found for SHA {DEMO_SHA}. Run Section 10 first.")
    else:
        bundle = load_json(bundle_path)
        system_prompt = bundle['_meta']['llm_system_prompt']
        user_content = json.dumps({k: v for k, v in bundle.items() if k != '_meta'}, indent=2)

        message = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1500,
            system=system_prompt,
            messages=[{"role": "user", "content": user_content}]
        )
        analysis = json.loads(message.content[0].text)
        print(json.dumps(analysis, indent=2))
        save_json(analysis, LLM_DIR / f"{DEMO_SHA}_analysis.json")
else:
    print("DEMO_SHA not set — skipping demo LLM call.")

In [ ]:
# ── Bulk LLM analysis (run after verifying demo above) ────────────────────
# Processes all bundles that don't yet have a *_analysis.json.
# Uses a simple rate-limited loop — add concurrency (asyncio) for speed.

RUN_BULK_LLM = False  # Set True only after testing demo above

if RUN_BULK_LLM:
    import anthropic
    client = anthropic.Anthropic()

    bundle_paths = sorted(LLM_DIR.glob("*.json"))
    bundle_paths = [p for p in bundle_paths if not p.stem.endswith("_analysis")]
    already_analysed = {p.stem.replace("_analysis", "") for p in LLM_DIR.glob("*_analysis.json")}
    todo = [p for p in bundle_paths if p.stem not in already_analysed]

    print(f"Commits to analyse: {len(todo)}")

    for bundle_path in tqdm(todo, desc="🤖 LLM analysis"):
        try:
            bundle = load_json(bundle_path)
            system_prompt = bundle['_meta']['llm_system_prompt']
            user_content = json.dumps({k: v for k, v in bundle.items() if k != '_meta'}, indent=2)

            message = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1500,
                system=system_prompt,
                messages=[{"role": "user", "content": user_content}]
            )
            raw_text = message.content[0].text
            # Strip markdown fences if present
            clean = raw_text.replace("```json", "").replace("```", "").strip()
            analysis = json.loads(clean)
            analysis['_sha'] = bundle_path.stem
            analysis['_repo'] = bundle.get('repo', '')
            analysis['_date'] = bundle.get('author', {}).get('git_date', '')

            save_json(analysis, LLM_DIR / f"{bundle_path.stem}_analysis.json")
            time.sleep(0.3)  # gentle rate limiting

        except Exception as e:
            print(f"  ⚠️  Error on {bundle_path.stem}: {e}")

    print("✅ Bulk LLM analysis complete.")

## Section 11 — Court-Ready Summary Report

In [ ]:
# Reload all data
commits_df = pd.read_csv(CHECKPOINT_COMMITS)
commits_df['git_author_date'] = pd.to_datetime(commits_df['git_author_date'], utc=True, errors='coerce')
repos_df = pd.read_csv(CHECKPOINT_REPOS)

issues_path = OUTPUT_DIR / "issues" / "all_issues.csv"
issues_df = pd.read_csv(issues_path) if issues_path.exists() else pd.DataFrame()

gist_path = OUTPUT_DIR / "gists" / "all_gists.csv"
gists_df = pd.read_csv(gist_path) if gist_path.exists() else pd.DataFrame()

events_df_path = OUTPUT_DIR / "events" / "account_events.csv"
events_df = pd.read_csv(events_df_path) if events_df_path.exists() else pd.DataFrame()

# ── Build report ──────────────────────────────────────────────────────────
report_lines = []
def h(text, level=1): report_lines.append("#" * level + " " + text)
def p(text): report_lines.append(text)
def rule(): report_lines.append("---")
def nl(): report_lines.append("")

h(f"GitHub Account Audit — {TARGET.login}")
p(f"*Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}*")
nl()
p("> This document is a machine-generated audit of GitHub account activity. "
  "All timestamps are UTC. All data sourced from the GitHub REST API.")
rule()

h("1. Account Profile", 2)
snap = load_json(OUTPUT_DIR / "account_snapshot.json")
for k, v in snap.items():
    if v is not None and not k.startswith('audit'):
        p(f"- **{k}**: {v}")
nl()

h("2. Repository Summary", 2)
p(f"- Total repositories: **{len(repos_df)}**")
p(f"- Own (non-fork): {len(repos_df[~repos_df['fork']])}")
p(f"- Forks: {len(repos_df[repos_df['fork']])}")
p(f"- Private: {len(repos_df[repos_df['private']])}")
p(f"- Archived: {len(repos_df[repos_df['archived']])}")
nl()
p("| Repository | Created | Language | Private | Archived |")
p("|---|---|---|---|---|")
for _, r in repos_df.sort_values('created_at').iterrows():
    p(f"| {r['repo_full_name']} | {str(r['created_at'])[:10]} | {r['primary_language']} | {r['private']} | {r['archived']} |")
nl()

h("3. Commit Statistics", 2)
active_commits = commits_df.dropna(subset=['git_author_date'])
p(f"- Total commits: **{len(active_commits)}**")
p(f"- First commit: {active_commits['git_author_date'].min()}")
p(f"- Latest commit: {active_commits['git_author_date'].max()}")
p(f"- Unique repos committed to: {active_commits['repo'].nunique()}")
p(f"- Total lines added: {active_commits['additions'].sum():,}")
p(f"- Total lines deleted: {active_commits['deletions'].sum():,}")
p(f"- Unique commit email addresses: {active_commits['git_author_email'].nunique()}")
nl()

h("4. Issues & Pull Requests", 2)
if not issues_df.empty:
    p(f"- Total issues/PRs created: {len(issues_df)}")
    if 'type' in issues_df.columns:
        for t, cnt in issues_df['type'].value_counts().items():
            p(f"  - {t}: {cnt}")
nl()

h("5. Gists", 2)
p(f"- Total gists: {len(gists_df)}")
nl()

h("6. Spam Investigation Findings", 2)
p("The following patterns were analysed to identify potential spam flag triggers:")
nl()
if not active_commits.empty:
    cpd = active_commits.groupby(active_commits['git_author_date'].dt.date).size()
    mean_c = cpd.mean()
    std_c = cpd.std()
    burst_days = cpd[cpd > mean_c + 2 * std_c]
    p(f"**Commit burst days** (>2σ above mean):")
    if not burst_days.empty:
        for day, cnt in burst_days.items():
            p(f"  - {day}: {cnt} commits")
    else:
        p("  None detected.")
nl()
p("**Note:** GitHub does not expose the spam-flag event via API. "
  "The above analysis identifies statistical anomalies in activity patterns "
  "that are commonly associated with automated spam detection triggers.")

h("7. Data Provenance", 2)
p("All data collected from the GitHub REST API using PyGitHub.")
p(f"API authenticated as: `{me.login}`")
p(f"Audit script run at: {ts(datetime.now(timezone.utc))}")
p(f"Files produced:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        p(f"  - `{f.relative_to(OUTPUT_DIR)}` ({size_kb:.1f} KB)")

# Write the report
report_text = "\n".join(report_lines)
report_path = OUTPUT_DIR / "COURT_SUMMARY.md"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(f"✅ Court summary written to {report_path}")
print(f"   Length: {len(report_text):,} chars")

---
## Output Directory Structure

```
github_audit/
├── account_snapshot.json        ← Account profile (court exhibit A)
├── repos.csv                    ← All repositories with metadata
├── organizations.csv            ← Org memberships
├── activity_analysis.png        ← Commit pattern charts
├── COURT_SUMMARY.md             ← Human-readable court overview
├── commits/
│   ├── all_commits.csv          ← Every commit with stats
│   └── done_repos.json          ← Checkpoint tracker
├── diffs/
│   └── <sha>.json               ← File diffs per commit
├── events/
│   ├── account_events.json      ← Full event stream
│   └── account_events.csv
├── gists/
│   ├── all_gists.json
│   └── all_gists.csv
├── issues/
│   ├── all_issues.csv           ← Issues & PRs
│   ├── all_issue_comments.csv
│   └── all_commit_comments.csv
└── llm_bundles/
    ├── <sha>.json               ← Self-contained commit bundle for LLM
    └── <sha>_analysis.json      ← LLM output (after Section 10)
```

**Next steps for LLM pipeline:**
1. Test one commit with `DEMO_SHA` in Section 10
2. Set `RUN_BULK_LLM = True` for full batch  
3. Consolidate all `*_analysis.json` into a single timeline document